# ASL Sign Recognition — Kaggle GPU training

Runs the full Phase-1 pipeline on a Kaggle GPU: build landmark cache → signer-disjoint split → train.

**Before you run (two one-time setup steps):**
1. **Enable the GPU** — right sidebar → *Settings* → *Accelerator* → **GPU T4 x2** (or P100).
2. **Add the data** — *Add Data* → search **"Google - Isolated Sign Language Recognition"** (`asl-signs`) → Add. It mounts at `/kaggle/input/asl-signs`.
3. **Set `REPO_URL`** in the first code cell to your GitHub repo. (No GitHub? See the note below the cell.)

Expected: ~30 sec/epoch on a T4 (vs ~5 min on a laptop). A full 80-epoch run is well under an hour.

In [ ]:
import os, subprocess

# >>> SET THIS to your repo URL <<<
REPO_URL = "https://github.com/arghavan-b/asl-sign-recognition.git"
DATA_DIR = "/kaggle/input/asl-signs"          # competition data (Add Data → asl-signs)
REPO_DIR = "/kaggle/working/asl-sign-recognition"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "pyyaml"], check=False)
print("cwd:", os.getcwd())
print("data exists:", os.path.exists(os.path.join(DATA_DIR, "train.csv")))

*No GitHub repo?* Zip your local `asl-sign-recognition` folder, upload it via **Add Data → Upload**, then set `REPO_DIR` to its `/kaggle/input/...` path and skip the `git clone` (copy it into `/kaggle/working` first if you want it writable).

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU — enable it in Settings → Accelerator, then restart.")

## 1. Build the landmark cache
Converts the parquet landmarks into `data/landmarks/<SIGN>/<id>.npy`. One-time, ~15–30 min on Kaggle CPU. **Tip:** to avoid redoing this every session, *Save Version* with output, or save `data/landmarks` as a Kaggle Dataset and attach it next time.

In [ ]:
# Clean any partial/full-size output from a failed run (frees the disk).
!rm -rf data/landmarks

# --no-face -> 225-d (hands+pose) ~3.3GB instead of full 1629-d ~23GB, which
# overflows Kaggle's 20GB working disk. Matches training (use_face=false).
# (Add --max-per-sign 200 to cap, or --vocab configs/daily_glosses.txt to filter.)
!python scripts/adapt_google_islr.py --data-dir $DATA_DIR --out data/landmarks --no-face
!echo '--- disk ---' && df -h /kaggle/working | tail -1
!echo '--- clips ---' && find data/landmarks -name '*.npy' | wc -l

## 1b. (Optional) Save the landmark cache for reuse
Conversion takes 15–30 min and isn't persisted by default. Pack `data/landmarks` into a **single tar** (94k loose files choke Kaggle's output saver, one archive doesn't), then register it as a **Dataset** to skip reconversion next session.

After this runs: *Output* tab → find `landmarks.tar` → *New Dataset*. Next session, *Add Data* → your dataset → and run the **Reuse** cell instead of converting.

In [ ]:
# Pack into one uncompressed tar (landmarks are floats — compression barely helps,
# and uncompressed is much faster). ~3.3GB, fits the 20GB working dir.
!cd data && tar -cf /kaggle/working/landmarks.tar landmarks
!ls -lh /kaggle/working/landmarks.tar

# --- Reuse next session (skip conversion) ---
# 1. Add Data -> your saved landmarks dataset (mounts at /kaggle/input/<name>)
# 2. Uncomment and set the path:
# !mkdir -p data && tar -xf /kaggle/input/<your-landmarks-dataset>/landmarks.tar -C data
# !find data/landmarks -name '*.npy' | wc -l

## 2. Signer-disjoint split (the honest metric)
Splits by `participant_id` so validation measures generalization to signers not seen in training. `src.train` picks these files up automatically.

In [ ]:
!python scripts/make_participant_split.py \
    --data-dir $DATA_DIR --landmark-dir data/landmarks \
    --splits-out data/splits --val-frac 0.2

## 3. Train
Uses `configs/default.yaml` (GRU, hands+pose, augmentation on). `device: auto` selects CUDA. On a Linux GPU box you can raise data-loading parallelism — but the in-RAM cache already serves from one process, so it's fine as-is.

In [ ]:
!python -m src.train --config configs/default.yaml

## 4. Push for higher accuracy (optional)
With a GPU the ~30 sec/epoch turnaround makes these cheap to try. Each writes a temp config and trains into its own checkpoint dir.

In [ ]:
import yaml, copy
base = yaml.safe_load(open("configs/default.yaml"))

def run(name, overrides):
    cfg = copy.deepcopy(base)
    for dotted, val in overrides.items():
        node = cfg
        keys = dotted.split(".")
        for k in keys[:-1]:
            node = node.setdefault(k, {})
        node[keys[-1]] = val
    cfg["train"]["checkpoint_dir"] = f"checkpoints/{name}"
    path = f"/kaggle/working/cfg_{name}.yaml"
    yaml.safe_dump(cfg, open(path, "w"))
    print(f"=== {name}: {overrides} ===")
    os.system(f"python -m src.train --config {path}")

# Bigger GRU (model isn't saturating train → more capacity should lift both curves):
run("gru_big", {"model.hidden_dim": 384, "model.num_layers": 3})

# Transformer (what won this competition; worth it now that data is large):
# run("transformer", {"model.type": "transformer", "model.num_layers": 4, "model.hidden_dim": 256})

## 5. Get your model out
Checkpoints land in `checkpoints/` under `/kaggle/working`, which you can download from the *Output* tab (or *Save Version* to persist).

In [ ]:
!ls -lh checkpoints* 2>/dev/null; find . -name 'best.pt' -exec ls -lh {} \;